<table  align="left" width="100%"> <tr>
        <td  style="background-color:#ffffff;"><a href="https://qworld.net" target="_blank"><img src="../images/qworld/qworld.jpg" width="35%" align="left"></a></td>
        <td  align="right" style="background-color:#ffffff;vertical-align:bottom;horizontal-align:right">
            prepared by Adam Glos and Özlem Salehi Köken
        </td>        
</tr></table>

#  Grover Algorithm for Max-Cut Problem

Finally, we are ready to implement the Grover algorithm for the Max-Cut problem. 

For the initialization, we will apply Hadamard gate to the qubits representing the color of each vertex. We know how to implement the Grover diffusion operator. Now we need to implement an oracle which will "mark" the colorings in which there are $k$ edges whose endpoints are colored using different colors.

For the oracle, the procedure goes as follows:

* Implement edge checking for each edge (check whether an edge has endpoints with different colors).
* Sum the outputs of edge checking step.
* Check whether the sum is equal to (or greater) $k$ and save the flag on the auxillary qubit.
* Apply $Z$ gate on the auxillary qubit.

The oracle works for the decision version of the problem which checks whether there exists a coloring such that there are at least $k$ edges whose endpoints are colored using different colors. How to find the maximum size for the cut?

Suppose that $k$ is a three-digit number $b_2b_1b_0$. First, we should check whether there is a coloring such that $b_2$ is equal to 1. If it is, then we check if there is a coloring such that $b_2=1$ and $b_1=1$. Otherwise, we should check for the coloring such that $b_2=0$ and $b_1$=1. This way, by doing the binary search over binary representation, we can finally find the cut of maximal size.

Let us consider the following graph.
<img src="../images/path3.png" width="25%" align="center">

We will check whether there is a coloring such that two edges connect vertices with a different color. To check that we will verify, whether for number $b_1b_0$, $b_1$ is set to 1. We will use:
* Three qubits (0-2) for vertices.
* Two qubits (3-4) for edges.
* Two qubits (5-6) for storing the sum.
* Single qubit (7) for the auxillary qubit.

Since qubit 7 stores the flag, that is whether $b_1$ is set to 1 or not, we will apply the $Z$ gate on that qubit.

Below we present the final Grover algorithm. Note that there are two such solutions, so we apply the oracle only once.

In [3]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZGate
from qiskit_aer import AerSimulator

def oracle_computation():
    temp_qc = QuantumCircuit(8)
    #edge checking
    temp_qc.cx(0, 3)
    temp_qc.cx(1, 3)
    temp_qc.cx(1, 4)
    temp_qc.cx(2, 4)
    
    #adding qubit 3
    temp_qc.cx(3,5)
    
    #adding qubit 4
    temp_qc.ccx(4, 5, 6)
    temp_qc.cx(4, 5)
    
    #check if b1 is equal to 1 and store the result in the auxillary qubit
    temp_qc.cx(6, 7)

    return temp_qc

def oracle(qc):
    qc.append(oracle_computation(), range(8))
    qc.z(7)
    qc.append(oracle_computation().inverse(), range(8))

def grover_diffusion(qc):
    qc.h(range(3))
    qc.x(range(3))
    qc.append(ZGate().control(2), range(3))
    qc.x(range(3))
    qc.h(range(3))


# the Grover algorithm
qc = QuantumCircuit(8, 3)

qc.h(range(3))

for i in range(1):
    oracle(qc)
    grover_diffusion()

qc.measure(range(3), range(3))

trials = 10000

tqc = transpile(qc, AerSimulator())
job = AerSimulator().run(tqc, shots=trials)
counts = job.result().get_counts(tqc)


print("Random guess probability:", 1./2**3)

for state, c in counts.items():
    print("Probability of obsering", state, ": " ,c/trials)

Random guess probability: 0.125
Probability of obsering 101 :  0.5104
Probability of obsering 010 :  0.4896


We observe that the Grover's Search outputs 101 and 010, which are the two colorings where there are 2 edges connecting vertices with different colors. 

In the remaining of this notebook, you will apply the same algorithm for other graphs. 

### Task 1

Implement the Grover algorithm for the following graph
<img src="../images/completebipartite.png" width="25%" align="center">

and check whether there is a coloring in which there are exactly four edges connecting vertices with a different color. Apply the oracle twice.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZGate
from qiskit_aer import AerSimulator

# 0-3: vertices
# ?-?: edge checking
# ?-?: the number
# ?-?: auxillary

def oracle_computation():
    #
    # your solution goes here
    # 

def oracle():    
    #
    # your solution goes here
    # 
    
def grover_diffusion():
    #
    # your solution goes here
    # 

# the Grover algorithm
qc = QuantumCircuit(12, 4)
qc.h(range(4))

for i in range(2):
    oracle(qc)
    grover_diffusion(qc)

qc.measure(range(4), range(4))

trials = 10000

tqc = transpile(qc, AerSimulator())
job = AerSimulator().run(tqc, shots=trials)
counts = job.result().get_counts(tqc)


print("Random guess probability:", 1./2**4)

for state, c in counts.items():
    print("Probability of obsering", state, ": " ,c/trials)

[click for our solution](04_grover_algorithm_for_max-cut_problem_solutions.ipynb#task1)

### Task 2

Implement the Grover Algorithm for the following graph
<img src="../images/finalgrover1.png" width="25%" align="center">

and check whether there is coloring with six or more edges connecting vertices with a different color. Apply the oracle twice. 

*Hint*: Note that you may check only two bits of the number.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import ZGate
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

#
# your solution
# 

trials = 10000

tqc = transpile(qc, AerSimulator())
job = AerSimulator().run(tqc, shots=trials)
counts = job.result().get_counts(tqc)


print("Random guess probability:", 1./2**5)

plot_histogram(counts)

[click for our solution](04_grover_algorithm_for_max-cut_problem_solutions.ipynb#task2)

## Final remarks

We have successfully implemented the Grover's Search algorithm for the Max-Cut problem. Congratulations! However, there are several interesting additional facts.

Currently for each edge, we have a separate qubit. We could reuse a single qubit at the cost of additional quantum gates. While the circuit would be longer, we could save some qubits.

Next, for each task, the number of iterations was given. While for bipartiteness checking we know there are always exactly two answers, this may not be the case for the Max-Cut problem. Fortunately, there is a *quantum counting algorithm*, which can efficiently determine the number of solutions and hence the number of iterations.